# Random Nucleotide Sampling

![Random Nucleotide Sampling](https://proto-bio.github.io/proto-assets/images/tool/random_nucleotide/hero.png)

This notebook demonstrates `run_random_nucleotide_sample`, which generates nucleotide sequence variants by filling masked positions with bases drawn from an IUPAC substitution scheme. It covers pre-masked sequences, automatic position selection via masking strategies, IUPAC scheme comparisons, RNA auto-detection, and batch library generation.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("random_nucleotide")
display_overview("random_nucleotide")
display_docs_section("random_nucleotide", "Background")

## Available tools

In [ ]:
display_available_tools("random_nucleotide")

### `run_random_nucleotide_sample`

`run_random_nucleotide_sample` fills masked positions in DNA or RNA sequences with bases drawn from an IUPAC substitution scheme. Positions marked with `_` are replaced by a randomly sampled base; all other positions are preserved unchanged. The masking strategy system lets you automatically select which positions to diversify — by count, fraction, or specific indices — or you can pre-mask positions manually. The tool auto-detects DNA versus RNA by the presence of uracil and converts sampled thymine bases to uracil for RNA inputs.

In [ ]:
from proto_tools import RandomNucleotideSampleInput, RandomNucleotideSampleConfig, run_random_nucleotide_sample
from proto_tools.transforms.masking import MaskingStrategy

In [ ]:
# Display input docs
display_api_reference("random_nucleotide", "input", "run_random_nucleotide_sample")

# Underscore characters mark positions for random substitution
inputs = RandomNucleotideSampleInput(sequences=["ACGT_CGT_A"])

In [ ]:
# Display config docs
display_api_reference("random_nucleotide", "config", "run_random_nucleotide_sample")

# Use a masking strategy to automatically select 2 positions to mutate | see docs above for all fields
config = RandomNucleotideSampleConfig(
    masking_strategy=MaskingStrategy(num_mutations=2),
    seed=42,
)

In [ ]:
# Run the sampling function
wild_type = "ACGTACGTAC"
result = run_random_nucleotide_sample(
    RandomNucleotideSampleInput(sequences=[wild_type]),
    config,
)

In [ ]:
# Display output docs
display_api_reference("random_nucleotide", "output", "run_random_nucleotide_sample")

# Show the variant and which positions were mutated
variant = result.sequences[0]
diffs = [i for i, (a, b) in enumerate(zip(wild_type, variant)) if a != b]
print(f"Wild type:         {wild_type}")
print(f"Variant:           {variant}")
print(f"Mutated positions: {diffs}")

# Demonstrate pre-masked input
result_premask = run_random_nucleotide_sample(RandomNucleotideSampleInput(sequences=["ACGT_CGT_A"]))
print(f"\nPre-masked input:  ACGT_CGT_A")
print(f"Variant:           {result_premask.sequences[0]}")

# Demonstrate IUPAC substitution schemes
print("\nIUPAC scheme comparison (all-masked sequence):")
for scheme in ["N", "R", "Y", "S", "W"]:
    scheme_config = RandomNucleotideSampleConfig(substitution_scheme=scheme, seed=42)
    scheme_result = run_random_nucleotide_sample(
        RandomNucleotideSampleInput(sequences=["____"]),
        scheme_config,
    )
    print(f"  {scheme}: {scheme_result.sequences[0]}")

# Demonstrate RNA auto-detection
rna_result = run_random_nucleotide_sample(RandomNucleotideSampleInput(sequences=["AUGC_CGU"]))
print(f"\nRNA input:    AUGC_CGU")
print(f"RNA variant:  {rna_result.sequences[0]}  (T converted to U)")

## Export Results

Sampling results can be saved to FASTA, TXT, or JSON format for downstream analysis or synthesis ordering.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Generate a batch of 5 variants from the same wild-type sequence
batch_inputs = RandomNucleotideSampleInput(sequences=["ACGTACGTAC"] * 5)
batch_config = RandomNucleotideSampleConfig(masking_strategy=MaskingStrategy(num_mutations=2))
batch_result = run_random_nucleotide_sample(batch_inputs, batch_config)

batch_result.export(name="random_nucleotide_variants", export_path=output_dir, file_format="fasta")
print(f"Exported sequences to {output_dir / 'random_nucleotide_variants.fasta'}")